# Comparing methods with `neuro_rave.evaluation`

This notebook fits several models on the same time-series data and scores them with the same held-out
checks:

- **reconstruction**: how well each model reconstructs new samples, relative to predicting the training mean;
- **domain predictability**: how well a separate classifier predicts the domain (session) from the embedding;
- **class information**: how well classes can be predicted, per domain, where true labels exist.

Each adversarial model is compared with **the same model with its adversary switched off**
(`domain_weight=0`). A lower domain-probe score alone does not show that a model corrected anything
biologically meaningful, and a remaining domain signal is not by itself a failure: some differences between
recordings can be real. Read the scores together.

The data are generated, so the notebook runs anywhere on CPU in a few minutes. Replace the data cell to use
your own recordings.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np

import neuro_rave
from neuro_rave import RAVE, VAEDANN, RAVEPlus
from neuro_rave import evaluation as ev

print("python    :", sys.executable)
print("neuro_rave:", neuro_rave.__version__, "from", neuro_rave.__file__)

## 1. Data

Replace this cell with your own data, keeping the variable names:

| Variable | Shape / content |
|---|---|
| `X_train`, `X_test` | `(samples, time)` or `(samples, time, channels)`; aligned time series on a shared time grid, scaled with training statistics |
| `domains_train`, `domains_test` | one domain ID (e.g. session) per sample |
| `y_train` | one class label per training sample, `None` where unknown (or set `y_train = None` if you have no labels) |
| `y_test` | true labels of test samples for scoring, `None` where unknown (or `y_test = None`) |

Hold out whole recordings for `X_test` where you can; scores on samples from recordings the model has seen
are optimistic. Here the generated data have three sessions that differ by gain, offset, a small time shift
and a slow drift; class labels are hidden in session `C` during training, and its true labels are used only
for scoring.

In [ ]:
T, CHANNELS = 60, 2
t = np.linspace(0.0, 1.0, T)
TEMPLATES = {
    "transient": np.exp(-(((t - 0.25) / 0.05) ** 2)),
    "sustained": 1 / (1 + np.exp(-(t - 0.2) / 0.02)) - 1 / (1 + np.exp(-(t - 0.7) / 0.02)),
    "oscillating": np.sin(2 * np.pi * 4 * t) * np.exp(-(((t - 0.5) / 0.25) ** 2)),
}
# gain, offset, time shift (samples), phase of a slow drift
SESSION_EFFECTS = {"A": (1.0, 0.0, 0, 0.0), "B": (1.2, 0.15, 1, 2.0), "C": (0.85, -0.1, -1, 4.0)}


def simulate(n_per_session, rng):
    series, sessions, classes = [], [], []
    for session, (gain, offset, shift, phase) in SESSION_EFFECTS.items():
        drift = 0.15 * np.cos(2 * np.pi * t + phase)
        for label in rng.choice(list(TEMPLATES), size=n_per_session):
            base = TEMPLATES[label] * rng.uniform(0.7, 1.3)
            channels = np.stack([base, 0.6 * np.roll(base, 3)], axis=-1)
            distorted = gain * np.roll(channels, shift, axis=0) + offset + drift[:, None]
            series.append(distorted + 0.35 * rng.standard_normal((T, CHANNELS)))
            sessions.append(session)
            classes.append(label)
    return np.asarray(series), np.asarray(sessions), np.asarray(classes, dtype=object)


rng = np.random.default_rng(0)
X_train, domains_train, classes_train = simulate(200, rng)
X_test, domains_test, y_test = simulate(200, rng)  # new samples from the same sessions

mean, std = X_train.mean(axis=0), X_train.std(axis=0) + 1e-8
X_train, X_test = (X_train - mean) / std, (X_test - mean) / std
y_train = [None if d == "C" else c for d, c in zip(domains_train, classes_train, strict=True)]

print(
    "X_train",
    X_train.shape,
    "| X_test",
    X_test.shape,
    "| domains",
    np.unique(domains_train).tolist(),
)

## 2. Models

Each adversarial model is paired with its zero-adversary control. Models that use class labels are only fitted
when `y_train` has labels; all models see the same samples, domains and labels. `VAEDANN` is an experimental
baseline (see the README for where it was tested). Several seeds show how much results vary between fits.

In [ ]:
SEEDS = [0, 1, 2]
has_labels = y_train is not None and any(v is not None for v in y_train)

MODELS = {
    "RAVE, adversary off": lambda s: RAVE(domain_weight=0.0, seed=s),
    "RAVE": lambda s: RAVE(seed=s),
    "VAEDANN, adversary off": lambda s: VAEDANN(domain_weight=0.0, seed=s),
    "VAEDANN": lambda s: VAEDANN(seed=s),
}
if has_labels:
    MODELS.update(
        {
            "RAVE+, adversary off": lambda s: RAVEPlus(domain_weight=0.0, seed=s),
            "RAVE+": lambda s: RAVEPlus(seed=s),
            "VAEDANN + labels, adversary off": lambda s: VAEDANN(domain_weight=0.0, seed=s),
            "VAEDANN + labels": lambda s: VAEDANN(seed=s),
        }
    )
uses_labels = {name: ("+" in name) for name in MODELS}

fitted = {}
for name, make in MODELS.items():
    for seed in SEEDS:
        model = make(seed)
        if uses_labels[name]:
            model.fit(X_train, domains_train, y=y_train)
        else:
            model.fit(X_train, domains_train)
        fitted[name, seed] = model
print(f"fitted {len(fitted)} models")

## 3. Scores on held-out samples

- `reconstruction_skill = 1 - SSE_model / SSE_reference`, where the reference predicts the **training mean**
  for every sample. 1 is perfect, 0 is no better than the mean, below 0 is worse (a collapsed fit).
- `domain_probe` trains a k-NN (and a linear) classifier on the training embeddings and scores it on the test
  embeddings. Balanced accuracy has chance level `1 / number of domains`.
- Class scores use the model's own predictions if it has a class head, and are reported per domain.

In [ ]:
domain_ids = np.unique(domains_test)
rows = []
for (name, seed), model in fitted.items():
    Z_train, Z_test = model.transform(X_train), model.transform(X_test)
    rec = ev.reconstruction_scores(
        X_test, model.reconstruct(X_test), reference_mean=X_train.mean(axis=0), groups=domains_test
    )
    knn = ev.domain_probe(Z_train, domains_train, Z_test, domains_test, kind="knn", seed=0)
    lin = ev.domain_probe(Z_train, domains_train, Z_test, domains_test, kind="linear", seed=0)
    row = {
        "model": name,
        "seed": seed,
        "skill": rec["reconstruction_skill"],
        "knn probe": knn["balanced_accuracy"],
        "linear probe": lin["balanced_accuracy"],
    }
    for d in domain_ids:
        row[f"skill {d}"] = rec["per_group"][d]["reconstruction_skill"]
    if getattr(model, "classes_", None) is not None and y_test is not None:
        cls = ev.classification_scores(y_test, model.predict(X_test), groups=domains_test)
        for d in domain_ids:
            row[f"class bal. acc. {d}"] = cls["per_group"][d]["balanced_accuracy"]
    rows.append(row)

print(f"chance (balanced) for the domain probe: {knn['chance_balanced_accuracy']:.2f}")
raw = ev.domain_probe(
    X_train.reshape(len(X_train), -1), domains_train, X_test.reshape(len(X_test), -1), domains_test
)
print(f"k-NN domain probe on the input itself: {raw['balanced_accuracy']:.3f}\n")


def show(columns, title):
    # mean and SD over seeds for each model; '-' where a score does not apply
    print(title)
    print(f"{'model':32s}" + "".join(f"{c:>22s}" for c in columns))
    for name in MODELS:
        cells = []
        for c in columns:
            values = [r[c] for r in rows if r["model"] == name and c in r]
            cells.append(f"{np.mean(values):.3f} ± {np.std(values):.3f}" if values else "-")
        print(f"{name:32s}" + "".join(f"{cell:>22s}" for cell in cells))
    print()


show(["skill", "knn probe", "linear probe"], "Overall")
show([f"skill {d}" for d in domain_ids], "Reconstruction skill per domain")
if any(k.startswith("class") for r in rows for k in r):
    show([f"class bal. acc. {d}" for d in domain_ids], "Class balanced accuracy per domain")

collapsed = [(r["model"], r["seed"]) for r in rows if r["skill"] <= 0]
print("\ncollapsed fits (skill <= 0):", collapsed or "none")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
markers = "osD^vP*X"
for i, name in enumerate(MODELS):
    pts = [(r["knn probe"], r["skill"]) for r in rows if r["model"] == name]
    off = "adversary off" in name
    ax.scatter(
        *zip(*pts, strict=True),
        marker=markers[i // 2 % len(markers)],
        s=45,
        label=name,
        facecolors="none" if off else f"C{i // 2}",
        edgecolors=f"C{i // 2}",
    )
ax.axvline(1 / len(domain_ids), color="0.5", ls=":", lw=1)
ax.axhline(0, color="0.5", ls=":", lw=1)
ax.set_xlabel("held-out k-NN domain probe, balanced accuracy (lower = less domain information)")
ax.set_ylabel("reconstruction skill vs. training mean")
ax.legend(fontsize=7, frameon=False, loc="lower left")
fig.tight_layout()
plt.show()

## 4. Reading the results

- **Compare each model with its own adversary-off fit.** A drop in the domain probe that costs most of the
  reconstruction skill is not a useful correction. On these data, RAVE lowers domain predictability strongly at
  little reconstruction cost; `VAEDANN` lowers it less. On other data the order can differ.
- **Look per domain.** An aggregate score can hide one badly reconstructed session.
- **Domain removal is partial.** No model here reaches chance on the k-NN probe, and that is not required.
- **Class scores in a domain without training labels** (session `C` here) are only measurable because the data
  are generated. With recordings you rarely have such ground truth; do not substitute cluster labels or
  anatomical proxies for it without saying so.
- **When it fails**: if sessions differ in ways a classifier can always read (large offsets, different sampling),
  an adversary may not reduce the domain probe at all; a large adversarial weight can then degrade or collapse the
  reconstruction (skill near or below 0). Lower the weight and compare again with the adversary-off fit.

## 5. Do classes from one domain match classes in another, consistently across fits?

`centroid_similarity` compares the mean embedding of each class in one set of samples with each class in another;
the two sides can use different label spaces. `best_matches` keeps the original IDs and reports margins, and
`matching_agreement` checks how stable the best matches are across seeds. This needs labels on both sides.

In [ ]:
if y_test is not None:
    reference, other = domain_ids[0], domain_ids[-1]
    y_arr = np.asarray(y_test, dtype=object)
    for name in ("RAVE", "VAEDANN"):
        results = []
        for seed in SEEDS:
            Z = fitted[name, seed].transform(X_test)
            a, b = domains_test == reference, domains_test == other
            results.append(ev.centroid_similarity(Z[a], y_arr[a], Z[b], y_arr[b]))
        best = ev.best_matches(
            results[0]["similarity"], results[0]["groups_a"], results[0]["groups_b"], top_k=2
        )
        agree = ev.matching_agreement(results, tie_tolerance=0.02)
        print(f"{name}: best matches of {reference}-classes in {other} (seed {SEEDS[0]}):")
        for m in best["matches"]:
            print(f"   {m['row']!s:12s} -> {m['top'][0]!s:12s} margin {m['margin']:.2f}")
        print(f"   agreement across {agree['n_fits']} seeds: {agree['pairwise_agreement']:.2f}\n")

## 6. Choosing settings: a small grid search on validation data

This is a **teaching example with a short budget** (few epochs, small data), not a benchmark. The pattern is
what matters:

1. Split the data **once** into training, validation and test sets before searching (group whole recordings
   when you can). Here the generated data give independent samples, so a random split is fine.
2. Refit every candidate **from scratch** on the training set with the same seed; never reuse weights.
3. Score candidates on the **validation** set: reconstruction skill, the domain probe (trained on training
   embeddings), and class accuracy where labels exist. Keep the probe settings fixed across candidates.
4. Choose with an explicit, written-down trade-off. Here: keep candidates whose validation skill is at least 90% of
   the adversary-off control's, then take the lowest domain probe. Also look at the non-dominated candidates.
   Choosing only by the lowest domain probe rewards collapsed embeddings.
5. Evaluate the chosen setting **once** on the test set.

Results are saved after each candidate, so re-running the cell skips finished candidates of the same search
(the file name encodes the data, budget, seed, grid and package version). Replace `param_grid`,
`EPOCHS` and the data with your own; on real recordings use a realistic epoch budget and confirm the chosen
setting with several seeds.

Where to start the grid: the defaults (`domain_weight=0.2`, `adversary_input="sample"`,
`decoder_conditioning="none"`) are the setting tested most for this one-head model at its default budget. With
source labels on two recorded datasets, `adversary_input="mean"` together with `decoder_conditioning="domain"`
lowered the domain probe as much and cost less reconstruction skill, but without labels it barely lowered the
probe on one dataset, and it destabilized one of three fits on synthetic data with large offsets (see the README).
No single weight or epoch budget is right for every dataset: the RGC-to-SC two-head setup in `examples/rgc2sc/`
was tuned at 1000 epochs, where weight 2 was selected from the tested weights; that is a research configuration for
those recordings, not a default.

In [ ]:
import hashlib
import itertools
import json
import tempfile
import time
from pathlib import Path

param_grid = {
    "domain_weight": [0.0, 0.2, 2.0],
    "adversary_input": ["sample", "mean"],
    "decoder_conditioning": ["none", "domain"],
}
EPOCHS, SEED = 40, 0
labels_for_fit = y_train if has_labels else None

# one fixed split of the training data into fitting and validation parts (test data stay untouched)
split_rng = np.random.default_rng(1)
is_val = split_rng.random(len(X_train)) < 0.25
X_fit, X_val = X_train[~is_val], X_train[is_val]
d_fit, d_val = np.asarray(domains_train)[~is_val], np.asarray(domains_train)[is_val]
y_fit = (
    None
    if labels_for_fit is None
    else [v for v, m in zip(labels_for_fit, is_val, strict=True) if not m]
)
y_val = (
    None
    if labels_for_fit is None
    else [v for v, m in zip(labels_for_fit, is_val, strict=True) if m]
)

candidates = [
    dict(zip(param_grid, values, strict=True)) for values in itertools.product(*param_grid.values())
]
# One results file per search: its name depends on the data, budget, seed, grid and package
# version, so re-running continues this search but never reuses results of a different one.
search_id = hashlib.sha256(
    X_fit.tobytes() + json.dumps([EPOCHS, SEED, param_grid, neuro_rave.__version__]).encode()
).hexdigest()[:12]
results_file = (
    Path(tempfile.gettempdir()) / f"neuro_rave_grid_{search_id}.json"
)  # use your own folder
done = json.loads(results_file.read_text()) if results_file.exists() else {}
print(f"{len(candidates)} candidates; {len(done)} already finished in {results_file}")


def score(model):
    rec = ev.reconstruction_scores(
        X_val, model.reconstruct(X_val, domains=d_val), reference_mean=X_fit.mean(axis=0)
    )
    probe = ev.domain_probe(model.transform(X_fit), d_fit, model.transform(X_val), d_val, seed=0)
    out = {"val_skill": rec["reconstruction_skill"], "val_domain_probe": probe["balanced_accuracy"]}
    if model.classes_ is not None and y_val is not None:
        out["val_class_bal_acc"] = ev.classification_scores(y_val, model.predict(X_val))[
            "balanced_accuracy"
        ]
    return out


for params in candidates:
    key = json.dumps(params, sort_keys=True)
    if key in done:
        continue
    start = time.time()
    model = VAEDANN(**params, epochs=EPOCHS, seed=SEED)  # a fresh model for every candidate
    model.fit(X_fit, d_fit, y=y_fit)
    done[key] = {
        **score(model),
        "fit_seconds": round(time.time() - start, 1),
        "params": model.get_params(),
    }
    results_file.write_text(json.dumps(done, indent=1))  # saved after every candidate

rows = sorted(done.values(), key=lambda r: r["val_domain_probe"])
header = ("domain_weight", "adversary", "decoder", "val skill", "val probe", "class")
print("".join(f"{h:>14s}" for h in header))
for r in rows:
    p = r["params"]
    cells = (p["domain_weight"], p["adversary_input"], p["decoder_conditioning"])
    scores = (r["val_skill"], r["val_domain_probe"], r.get("val_class_bal_acc", float("nan")))
    print("".join(f"{c!s:>14s}" for c in cells) + "".join(f"{v:>14.3f}" for v in scores))

In [ ]:
def nondominated(rows):
    # higher skill and lower domain probe are both better
    return [
        r
        for r in rows
        if not any(
            o["val_skill"] >= r["val_skill"]
            and o["val_domain_probe"] <= r["val_domain_probe"]
            and o is not r
            and (o["val_skill"] > r["val_skill"] or o["val_domain_probe"] < r["val_domain_probe"])
            for o in rows
        )
    ]


control_skill = {
    (r["params"]["decoder_conditioning"]): r["val_skill"]
    for r in rows
    if r["params"]["domain_weight"] == 0
}
eligible = [
    r
    for r in rows
    if r["params"]["domain_weight"] > 0
    and r["val_skill"] >= 0.9 * control_skill[r["params"]["decoder_conditioning"]]
]
print(
    "non-dominated on validation:",
    [
        (
            r["params"]["domain_weight"],
            r["params"]["adversary_input"],
            r["params"]["decoder_conditioning"],
        )
        for r in nondominated(rows)
    ],
)
if eligible:
    chosen = min(eligible, key=lambda r: r["val_domain_probe"])["params"]
    print("chosen by the stated rule:", {k: chosen[k] for k in param_grid})
    # refit on train (= fit + val); chosen holds all settings, including epochs and seed
    final = VAEDANN(**chosen).fit(X_train, domains_train, y=labels_for_fit)
    test_probe = ev.domain_probe(
        final.transform(X_train), domains_train, final.transform(X_test), domains_test
    )
    test_rec = ev.reconstruction_scores(
        X_test, final.reconstruct(X_test, domains=domains_test), reference_mean=X_train.mean(axis=0)
    )
    print(
        f"test (evaluated once): skill {test_rec['reconstruction_skill']:.3f}, "
        f"domain probe {test_probe['balanced_accuracy']:.3f}"
    )
else:
    print(
        "no candidate kept 90% of its control's validation skill; widen the grid or relax the rule"
    )

## 7. Training history, checkpoints and resuming

`history_` holds per-epoch means of each loss term, the domain classifier's training accuracy and the reversal
strength `alpha`. For long fits, pass `checkpoint=` to save the full training state every `checkpoint_every`
epochs; after an interruption the same call with `resume=True` continues where it stopped (it refuses if the
settings or the data differ).

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    ckpt = Path(tmp) / "vaedann.ckpt"
    model = VAEDANN(epochs=EPOCHS, seed=0)
    model.fit(X_train, domains_train, checkpoint=ckpt, checkpoint_every=10)
    # ... after an interruption, the same call continues from the last saved epoch:
    model = VAEDANN(epochs=EPOCHS, seed=0).fit(X_train, domains_train, checkpoint=ckpt, resume=True)

fig, axes = plt.subplots(1, 3, figsize=(10, 2.8))
for ax, key in zip(axes, ("reconstruction", "domain", "alpha"), strict=True):
    ax.plot(model.history_[key])
    ax.set_title(key)
    ax.set_xlabel("epoch")
fig.tight_layout()
plt.show()